<a href="https://colab.research.google.com/github/meung0818/click_event/blob/main/%EA%B3%A7%EA%B0%90_%EA%B0%80%EA%B3%84%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 🐻 곧감 가계부
# Supabase + Gradio 버전
# ============================================================

# ============================================================
# 0. 필요한 라이브러리
# ============================================================

!pip install -q gradio supabase seaborn
!apt-get -qq install fonts-nanum


import gradio as gr
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

from supabase import create_client


# ============================================================
# 1. Supabase 연결
# ============================================================

SUPABASE_URL = "https://qadtyvmpwxzqirizcbfg.supabase.co"
SUPABASE_KEY = "sb_publishable_fxbYcC6Fyp-xYHEqR4TlPA_NWDJj8wr"

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

print("Supabase 연결 성공!")


# ============================================================
# 2. 한글 폰트
# ============================================================

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

try:
    fm.fontManager.addfont(font_path)

    font = fm.FontProperties(fname=font_path)
    font_name = font.get_name()

    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False

    print("한글 폰트 설정 완료!")

except Exception:
    print("한글 폰트를 찾지 못했습니다.")


# ============================================================
# 3. 현재 총 자산
# ============================================================

def get_total_asset():

    result = (
        supabase
        .table("assets")
        .select("money")
        .execute()
    )

    total = sum(
        row["money"]
        for row in result.data
    )

    return total


# ============================================================
# 4. 자산 현황 출력
# ============================================================

def show_assets():

    result = (
        supabase
        .table("assets")
        .select("name, money")
        .order("id")
        .execute()
    )

    rows = result.data

    result_text = ""

    for row in rows:

        result_text += (
            f"{row['name']} : "
            f"{row['money']:,}원\n"
        )

    result_text += "----------------------\n"

    result_text += (
        f"총 자산 : "
        f"{get_total_asset():,}원"
    )

    return result_text


# ============================================================
# 5. 자산 목록
# ============================================================

def get_asset_choices():

    result = (
        supabase
        .table("assets")
        .select("name")
        .order("id")
        .execute()
    )

    return [
        row["name"]
        for row in result.data
    ]


# ============================================================
# 6. 월 목록
# ============================================================

def get_month_choices():

    result = (
        supabase
        .table("transactions")
        .select("date")
        .order("date", desc=True)
        .execute()
    )

    months = set()

    for row in result.data:

        date = dt.date.fromisoformat(
            row["date"]
        )

        months.add(
            (
                date.year,
                date.month
            )
        )

    months = sorted(
        months,
        reverse=True
    )

    choices = ["전체"]

    for year, month in months:

        choices.append(
            f"{year}년 {month}월"
        )

    return choices


# ============================================================
# 7. 거래내역 표
# ============================================================

def get_transactions():

    result = (
        supabase
        .table("transactions")
        .select(
            "id, date, kind, category, asset_name, amount"
        )
        .order("id")
        .execute()
    )

    if not result.data:

        return pd.DataFrame(
            columns=[
                "번호",
                "날짜",
                "구분",
                "분류",
                "자산 종류",
                "금액"
            ]
        )

    data = []

    for i, row in enumerate(result.data):

        if row["kind"] == "수입":

            display_amount = (
                f"+{row['amount']:,}원"
            )

        else:

            display_amount = (
                f"-{row['amount']:,}원"
            )

        data.append(
            [
                i + 1,
                row["date"],
                row["kind"],
                row["category"],
                row["asset_name"],
                display_amount
            ]
        )

    return pd.DataFrame(
        data,
        columns=[
            "번호",
            "날짜",
            "구분",
            "분류",
            "자산 종류",
            "금액"
        ]
    )


# ============================================================
# 8. 삭제할 거래 목록
# ============================================================

def get_transaction_choices():

    result = (
        supabase
        .table("transactions")
        .select(
            "id, date, kind, category, asset_name, amount"
        )
        .order("id")
        .execute()
    )

    choices = []

    for row in result.data:

        if row["kind"] == "수입":
            sign = "+"

        else:
            sign = "-"

        label = (
            f"{row['date']} | "
            f"{row['kind']} | "
            f"{row['category']} | "
            f"{row['asset_name']} | "
            f"{sign}{row['amount']:,}원"
        )

        choices.append(
            (
                label,
                row["id"]
            )
        )

    return choices


# ============================================================
# 9. 거래 추가
# ============================================================

def add_transaction(
    kind,
    category,
    asset,
    amount
):

    # 분류 확인
    if not category or not category.strip():

        return (
            "⚠ 분류를 입력해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    # 금액 확인
    if amount is None or amount <= 0:

        return (
            "⚠ 올바른 금액을 입력해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    # 자산 확인
    if not asset:

        return (
            "⚠ 먼저 자산을 추가하고 선택해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    amount = int(amount)

    # 자산 조회
    asset_result = (
        supabase
        .table("assets")
        .select("money")
        .eq("name", asset)
        .execute()
    )

    if not asset_result.data:

        return (
            "⚠ 존재하지 않는 자산입니다.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    current_money = (
        asset_result.data[0]["money"]
    )

    # 수입 / 지출
    if kind == "수입":

        new_money = (
            current_money + amount
        )

    else:

        new_money = (
            current_money - amount
        )

    # 자산 업데이트
    (
        supabase
        .table("assets")
        .update(
            {
                "money": new_money
            }
        )
        .eq("name", asset)
        .execute()
    )

    # 오늘 날짜
    today = dt.date.today().isoformat()

    # 거래 저장
    (
        supabase
        .table("transactions")
        .insert(
            {
                "date": today,
                "kind": kind,
                "category": category.strip(),
                "asset_name": asset,
                "amount": amount
            }
        )
        .execute()
    )

    return (
        "✅ 거래가 저장되었습니다.",
        f"{get_total_asset():,}원",
        show_assets(),
        get_transactions(),
        gr.update(
            choices=get_transaction_choices(),
            value=None
        ),
        gr.update(
            choices=get_month_choices()
        )
    )


# ============================================================
# 10. 거래 삭제
# ============================================================

def delete_transaction(selected_id):

    if not selected_id:

        return (
            "⚠ 삭제할 거래를 선택해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    transaction_id = int(selected_id)

    # 거래 가져오기
    result = (
        supabase
        .table("transactions")
        .select(
            "kind, asset_name, amount"
        )
        .eq("id", transaction_id)
        .execute()
    )

    if not result.data:

        return (
            "⚠ 해당 거래를 찾을 수 없습니다.",
            f"{get_total_asset():,}원",
            show_assets(),
            get_transactions(),
            gr.update(
                choices=get_transaction_choices(),
                value=None
            ),
            gr.update(
                choices=get_month_choices()
            )
        )

    transaction = result.data[0]

    kind = transaction["kind"]
    asset_name = transaction["asset_name"]
    amount = transaction["amount"]

    # 자산 가져오기
    asset_result = (
        supabase
        .table("assets")
        .select("money")
        .eq("name", asset_name)
        .execute()
    )

    if asset_result.data:

        current_money = (
            asset_result.data[0]["money"]
        )

        # 원래 거래를 되돌림
        if kind == "수입":

            new_money = (
                current_money - amount
            )

        else:

            new_money = (
                current_money + amount
            )

        (
            supabase
            .table("assets")
            .update(
                {
                    "money": new_money
                }
            )
            .eq("name", asset_name)
            .execute()
        )

    # 거래 삭제
    (
        supabase
        .table("transactions")
        .delete()
        .eq("id", transaction_id)
        .execute()
    )

    return (
        "🗑 거래가 삭제되었습니다.",
        f"{get_total_asset():,}원",
        show_assets(),
        get_transactions(),
        gr.update(
            choices=get_transaction_choices(),
            value=None
        ),
        gr.update(
            choices=get_month_choices()
        )
    )


# ============================================================
# 11. 자산 추가
# ============================================================

def add_asset(
    asset_name,
    initial_money
):

    asset_name = (
        asset_name or ""
    ).strip()

    # 이름 확인
    if not asset_name:

        return (
            "⚠ 자산 이름을 입력해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 중복 확인
    existing = (
        supabase
        .table("assets")
        .select("id")
        .eq("name", asset_name)
        .execute()
    )

    if existing.data:

        return (
            "⚠ 이미 존재하는 자산입니다.",
            f"{get_total_asset():,}원",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    money = int(
        initial_money or 0
    )

    # 금액 확인
    if money < 0:

        return (
            "⚠ 초기 금액은 0원 이상으로 입력해주세요.",
            f"{get_total_asset():,}원",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 자산 추가
    (
        supabase
        .table("assets")
        .insert(
            {
                "name": asset_name,
                "money": money
            }
        )
        .execute()
    )

    choices = get_asset_choices()

    return (
        f"✅ '{asset_name}'이(가) 추가되었습니다.",
        f"{get_total_asset():,}원",
        show_assets(),
        gr.update(
            choices=choices,
            value=asset_name
        ),
        gr.update(
            choices=choices
        ),
        gr.update(
            choices=choices
        )
    )


# ============================================================
# 12. 자산 이름 변경
# ============================================================

def rename_asset(
    old_name,
    new_name
):

    old_name = (
        old_name or ""
    ).strip()

    new_name = (
        new_name or ""
    ).strip()

    # 기존 이름 확인
    if not old_name:

        return (
            "⚠ 변경할 자산을 선택해주세요.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 새 이름 확인
    if not new_name:

        return (
            "⚠ 새로운 이름을 입력해주세요.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 같은 이름
    if old_name == new_name:

        return (
            "⚠ 기존 이름과 같은 이름입니다.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 기존 자산 확인
    old_result = (
        supabase
        .table("assets")
        .select("id")
        .eq("name", old_name)
        .execute()
    )

    if not old_result.data:

        return (
            "⚠ 존재하지 않는 자산입니다.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 새 이름 중복 확인
    new_result = (
        supabase
        .table("assets")
        .select("id")
        .eq("name", new_name)
        .execute()
    )

    if new_result.data:

        return (
            "⚠ 이미 존재하는 자산 이름입니다.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 자산 이름 변경
    (
        supabase
        .table("assets")
        .update(
            {
                "name": new_name
            }
        )
        .eq("name", old_name)
        .execute()
    )

    # 기존 거래내역의 자산 이름도 변경
    (
        supabase
        .table("transactions")
        .update(
            {
                "asset_name": new_name
            }
        )
        .eq("asset_name", old_name)
        .execute()
    )

    choices = get_asset_choices()

    return (
        f"✅ '{old_name}' → '{new_name}'으로 변경되었습니다.",
        show_assets(),
        gr.update(
            choices=choices,
            value=new_name
        ),
        gr.update(
            choices=choices
        ),
        gr.update(
            choices=choices
        )
    )


# ============================================================
# 13. 자산 삭제
# ============================================================

def delete_asset(asset_name):

    if not asset_name:

        return (
            "⚠ 삭제할 자산을 선택해주세요.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 자산 존재 확인
    asset_result = (
        supabase
        .table("assets")
        .select("id")
        .eq("name", asset_name)
        .execute()
    )

    if not asset_result.data:

        return (
            "⚠ 존재하지 않는 자산입니다.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 거래내역 확인
    transaction_result = (
        supabase
        .table("transactions")
        .select("id")
        .eq("asset_name", asset_name)
        .execute()
    )

    if transaction_result.data:

        return (
            "⚠ 거래내역이 있는 자산은 삭제할 수 없습니다.",
            show_assets(),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            ),
            gr.update(
                choices=get_asset_choices()
            )
        )

    # 자산 삭제
    (
        supabase
        .table("assets")
        .delete()
        .eq("name", asset_name)
        .execute()
    )

    choices = get_asset_choices()

    return (
        f"🗑 '{asset_name}'이(가) 삭제되었습니다.",
        show_assets(),
        gr.update(
            choices=choices,
            value=choices[0] if choices else None
        ),
        gr.update(
            choices=choices,
            value=choices[0] if choices else None
        ),
        gr.update(
            choices=choices,
            value=choices[0] if choices else None
        )
    )


# ============================================================
# 14. 월별 통계
# ============================================================

def get_statistics(selected):

    result = (
        supabase
        .table("transactions")
        .select(
            "date, kind, category, amount"
        )
        .order("id")
        .execute()
    )

    rows = result.data

    if not rows:

        return (
            "아직 거래내역이 없습니다.",
            None
        )

    transactions_data = []

    for row in rows:

        transactions_data.append(
            {
                "날짜": dt.date.fromisoformat(
                    row["date"]
                ),
                "구분": row["kind"],
                "분류": row["category"],
                "금액": row["amount"]
            }
        )

    # 전체
    if selected == "전체":

        filtered_transactions = (
            transactions_data
        )

        title = "전체 통계"

    # 특정 월
    else:

        year = int(
            selected[:4]
        )

        month = int(
            selected[6:-1]
        )

        filtered_transactions = []

        for trans in transactions_data:

            if (
                trans["날짜"].year == year
                and
                trans["날짜"].month == month
            ):

                filtered_transactions.append(
                    trans
                )

        title = selected

    total_income = 0
    total_expense = 0

    expense_category = {}

    # 수입 / 지출 계산
    for trans in filtered_transactions:

        if trans["구분"] == "수입":

            total_income += (
                trans["금액"]
            )

        else:

            total_expense += (
                trans["금액"]
            )

            category = trans["분류"]

            if category not in expense_category:

                expense_category[category] = 0

            expense_category[category] += (
                trans["금액"]
            )

    variation = (
        total_income - total_expense
    )

    # 통계 텍스트
    result_text = ""

    result_text += (
        f"===== {title} =====\n\n"
    )

    result_text += (
        f"총 수입 : "
        f"{total_income:,}원\n"
    )

    result_text += (
        f"총 지출 : "
        f"{total_expense:,}원\n"
    )

    result_text += (
        "-----------------------------\n"
    )

    result_text += (
        f"순증감 : "
        f"{variation:,}원\n"
    )

    # 지출 내역
    result_text += (
        "\n[지출 내역]\n"
    )

    if expense_category:

        for category, amount in (
            expense_category.items()
        ):

            result_text += (
                f"{category} : "
                f"{amount:,}원\n"
            )

    else:

        result_text += (
            "지출 내역이 없습니다.\n"
        )

    # 가장 많이 쓴 항목
    result_text += (
        "\n[가장 많이 쓴 항목]\n"
    )

    if expense_category:

        max_category = max(
            expense_category,
            key=expense_category.get
        )

        result_text += (
            f"{max_category} : "
            f"{expense_category[max_category]:,}원\n"
        )

    else:

        result_text += (
            "지출 내역이 없습니다.\n"
        )

    # 지출 비율
    result_text += (
        "\n[지출 비율]\n"
    )

    if total_expense > 0:

        for category, amount in (
            expense_category.items()
        ):

            ratio = (
                amount
                / total_expense
                * 100
            )

            result_text += (
                f"{category} : "
                f"{ratio:.1f}%\n"
            )

    else:

        result_text += (
            "지출 내역이 없습니다."
        )

    # 그래프
    if expense_category:

        categories = list(
            expense_category.keys()
        )

        amounts = list(
            expense_category.values()
        )

        fig, ax = plt.subplots(
            figsize=(9, 5)
        )

        bars = ax.barh(
            categories,
            amounts,
            color="#FF8EA0"
        )

        ax.set_xlabel(
            "금액 (원)"
        )

        ax.set_ylabel(
            f"{title} 지출 내역"
        )

        max_amount = max(amounts)

        ax.set_xlim(
            0,
            max_amount * 1.25
        )

        for bar, amount in zip(
            bars,
            amounts
        ):

            ax.text(
                bar.get_width()
                + max_amount * 0.02,

                bar.get_y()
                + bar.get_height() / 2,

                f"{amount:,}원",

                va="center"
            )

        ax.grid(
            axis="x",
            linestyle="--",
            alpha=0.3
        )

        ax.spines[
            "top"
        ].set_visible(False)

        ax.spines[
            "right"
        ].set_visible(False)

        plt.tight_layout()

    else:

        fig = None

    return (
        result_text,
        fig
    )


# ============================================================
# 15. GUI
# ============================================================

custom_css = """

.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
}

.title-box {
    text-align: center;
    padding: 25px 10px 15px 10px;
}

.title-box h1 {
    font-size: 36px !important;
    margin-bottom: 5px !important;
}

.title-box p {
    font-size: 15px !important;
    opacity: 0.7;
}

.section-title {
    font-size: 22px !important;
    font-weight: 700 !important;
    margin-top: 10px !important;
    margin-bottom: 10px !important;
}

.asset-card {
    border-radius: 18px !important;
    padding: 20px !important;
    border: 1px solid #f0d9df !important;
}

.asset-view {
    border-radius: 15px !important;
}

button {
    border-radius: 10px !important;
}

.transaction-box {
    border-radius: 15px !important;
}

.statistics-box {
    border-radius: 15px !important;
}

hr {
    margin-top: 25px !important;
    margin-bottom: 25px !important;
}

"""


with gr.Blocks(
    title="곧감 가계부",
    css=custom_css
) as app:

    # ========================================================
    # 제목
    # ========================================================

    gr.Markdown(
        """
        <div class="title-box">

        # 🐻 곧감 가계부

        ### 곧감니다요 가계부

        </div>
        """
    )


    # ========================================================
    # 현재 총 자산
    # ========================================================

    gr.Markdown(
        "## 💰 현재 자산"
    )

    total_asset = gr.Textbox(
        label="총 자산",
        value=f"{get_total_asset():,}원",
        interactive=False
    )


    # ========================================================
    # 자산 현황
    # ========================================================

    gr.Markdown(
        "### 💳 자산 현황"
    )

    asset_view = gr.Textbox(
        label="",
        value=show_assets(),
        lines=6,
        interactive=False
    )


    # ========================================================
    # 자산 관리
    # ========================================================

    gr.Markdown("---")

    gr.Markdown(
        "## 🏦 자산 관리"
    )


    # 자산 추가
    gr.Markdown(
        "### ➕ 자산 추가"
    )

    with gr.Row():

        new_asset_name = gr.Textbox(
            label="자산 이름",
            placeholder="예: 국민은행"
        )

        new_asset_money = gr.Number(
            label="초기 금액",
            value=0
        )

    add_asset_button = gr.Button(
        "➕ 자산 추가"
    )

    asset_message = gr.Textbox(
        label="",
        interactive=False
    )


    # 자산 이름 변경
    gr.Markdown(
        "### ✏️ 자산 이름 변경"
    )

    with gr.Row():

        rename_old = gr.Dropdown(
            choices=get_asset_choices(),
            label="현재 이름"
        )

        rename_new = gr.Textbox(
            label="새 이름",
            placeholder="예: BNK 부산은행"
        )

    rename_button = gr.Button(
        "✏️ 이름 변경"
    )


    # 자산 삭제
    gr.Markdown(
        "### 🗑️ 자산 삭제"
    )

    delete_asset_select = gr.Dropdown(
        choices=get_asset_choices(),
        label="삭제할 자산"
    )

    delete_asset_button = gr.Button(
        "🗑️ 자산 삭제"
    )


    # ========================================================
    # 거래 입력
    # ========================================================

    gr.Markdown("---")

    gr.Markdown(
        "## 💳 거래 입력"
    )

    with gr.Row():

        kind = gr.Radio(
            ["수입", "지출"],
            label="구분",
            value="지출"
        )

        category = gr.Textbox(
            label="분류",
            placeholder="예: 식비, 용돈, 교통비"
        )


    asset_choices = get_asset_choices()

    asset = gr.Dropdown(
        choices=asset_choices,
        label="자산 종류",
        value=(
            asset_choices[0]
            if asset_choices
            else None
        )
    )

    amount = gr.Number(
        label="금액",
        minimum=1
    )

    save_button = gr.Button(
        "💾 거래 저장",
        variant="primary"
    )

    message = gr.Textbox(
        label="",
        interactive=False
    )


    # ========================================================
    # 거래 내역
    # ========================================================

    gr.Markdown(
        "## 📋 거래내역"
    )

    transaction_table = gr.Dataframe(
        headers=[
            "번호",
            "날짜",
            "구분",
            "분류",
            "자산 종류",
            "금액"
        ],
        value=get_transactions(),
        interactive=False
    )


    # ========================================================
    # 거래 삭제
    # ========================================================

    gr.Markdown(
        "### 🗑️ 거래 삭제"
    )

    delete_transaction_select = gr.Dropdown(
        choices=get_transaction_choices(),
        label="삭제할 거래"
    )

    delete_transaction_button = gr.Button(
        "🗑️ 선택한 거래 삭제"
    )

    delete_message = gr.Textbox(
        label="",
        interactive=False
    )


    # ========================================================
    # 월별 통계
    # ========================================================

    gr.Markdown("---")

    gr.Markdown(
        "## 📊 월별 통계"
    )

    with gr.Row():

        month_select = gr.Dropdown(
            choices=get_month_choices(),
            value="전체",
            label="확인할 기간"
        )

        stat_button = gr.Button(
            "📊 통계 보기",
            variant="primary"
        )


    statistics = gr.Textbox(
        label="통계 결과",
        lines=15,
        interactive=False
    )

    graph = gr.Plot(
        label="지출 그래프"
    )


    # ========================================================
    # 거래 저장 버튼 연결
    # ========================================================

    save_button.click(
        add_transaction,

        inputs=[
            kind,
            category,
            asset,
            amount
        ],

        outputs=[
            message,
            total_asset,
            asset_view,
            transaction_table,
            delete_transaction_select,
            month_select
        ]
    )


    # ========================================================
    # 거래 삭제 버튼 연결
    # ========================================================

    delete_transaction_button.click(
        delete_transaction,

        inputs=[
            delete_transaction_select
        ],

        outputs=[
            delete_message,
            total_asset,
            asset_view,
            transaction_table,
            delete_transaction_select,
            month_select
        ]
    )


    # ========================================================
    # 자산 추가 버튼 연결
    # ========================================================

    add_asset_button.click(
        add_asset,

        inputs=[
            new_asset_name,
            new_asset_money
        ],

        outputs=[
            asset_message,
            total_asset,
            asset_view,
            asset,
            rename_old,
            delete_asset_select
        ]
    )


    # ========================================================
    # 자산 이름 변경 버튼 연결
    # ========================================================

    rename_button.click(
        rename_asset,

        inputs=[
            rename_old,
            rename_new
        ],

        outputs=[
            asset_message,
            asset_view,
            asset,
            rename_old,
            delete_asset_select
        ]
    )


    # ========================================================
    # 자산 삭제 버튼 연결
    # ========================================================

    delete_asset_button.click(
        delete_asset,

        inputs=[
            delete_asset_select
        ],

        outputs=[
            asset_message,
            asset_view,
            asset,
            rename_old,
            delete_asset_select
        ]
    )


    # ========================================================
    # 통계 버튼 연결
    # ========================================================

    stat_button.click(
        get_statistics,

        inputs=[
            month_select
        ],

        outputs=[
            statistics,
            graph
        ]
    )


# ============================================================
# 16. 실행
# ============================================================

app.launch(
    share=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.3 MB/s eta 0:00:00
Selecting previously unselected package fonts-nanum.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...
Supabase 연결 성공!
한글 폰트 설정 완료!


/tmp/ipykernel_3649/2281852559.py:1293: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b69043c1ec1b772b8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
